# 00 — SAP Column Reference & Readable Naming Crosswalk

Interactive reference guide for translating SAP FI/CO column codes (`BUKRS`, `KOSTL`, `RCNTR`, `HKONT`, `RACCT`, `CPUDT`, `POPER`, `KSL`, etc.) into human-readable `UPPER_SNAKE` names—and converting back anytime.

### Codebase Naming Contract
- **Core `src/opex_forecast/` pipelines:** Keep `load_pharma_table(..., readable=False)` (the default) so production feature engineering, validation (`36/36`), and reconciliation stay on canonical SAP column names.
- **Notebooks, EDA & Executive Charts:** Pass `readable=True` to `load_pharma_table(table_name, readable=True)` or call `rename_to_readable(df, table_name)` for readable headers without modifying raw CSVs in `data/raw/pharma_v22/`.
- **Converting Back to SAP:** Call `rename_to_sap(df, table_name)` (table-scoped so ECC tables return `BUKRS`/`KOSTL`/`HKONT`/`MONAT` and S/4HANA `acdoca_universal_journal` returns `RBUKRS`/`RCNTR`/`RACCT`/`POPER`/`PS_PSP_PNR`).

In [1]:
import pandas as pd

from opex_forecast.column_map import (
    build_column_crosswalk,
    get_table_column_map,
    rename_to_readable,
    rename_to_sap,
)
from opex_forecast.config import DOCS_DIR, RAW_DATA_DIR
from opex_forecast.io import load_pharma_table

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 90)

## 1. Full Searchable SAP ↔ Readable Crosswalk (`206` Table-Column Pairs)

In [2]:
crosswalk = build_column_crosswalk()
print(f"Crosswalk rows: {len(crosswalk)} | Tables: {crosswalk['table_name'].nunique()} | Renamed SAP columns: {int(crosswalk['is_renamed'].sum())}")

# Filter to columns whose SAP name is translated to a more readable name
renamed_view = crosswalk[crosswalk["is_renamed"]][[
    "table_name", "sap_column", "readable_column", "dtype", "sap_reference", "ecc_equivalent", "ml_role"
]]
renamed_view.head(25)

Crosswalk rows: 206 | Tables: 14 | Renamed SAP columns: 112


,table_name,sap_column,readable_column,dtype,sap_reference,ecc_equivalent,ml_role
0,dim_company_code,BUKRS,COMPANY_CODE,string,T001-BUKRS,,HIERARCHY_KEY
1,dim_company_code,BUTXT,COMPANY_NAME,string,T001-BUTXT,,METADATA
2,dim_company_code,WAERS,LOCAL_CURRENCY,string,T001-WAERS,,CURRENCY_KEY
4,dim_plant,WERKS,PLANT_CODE,string,T001W-WERKS,,HIERARCHY_KEY
6,dim_plant,BUKRS,COMPANY_CODE,string,T001W/T001K-BUKRS,,HIERARCHY_KEY
10,dim_cost_center,BUKRS,COMPANY_CODE,string,CSKS-BUKRS,,HIERARCHY_KEY
11,dim_cost_center,WERKS,PLANT_CODE,string,Plant Link,,HIERARCHY_KEY
12,dim_cost_center,KOSTL,COST_CENTER_ID,string,CSKS-KOSTL,,ENTITY_KEY
13,dim_cost_center,KOSTL_NAME,COST_CENTER_NAME,string,CSKT-LTEXT,,METADATA
15,dim_cost_center,KOSTL_GROUP,COST_CENTER_GROUP,string,,,HIERARCHY_ATTRIBUTE


In [3]:
# Quick lookup helper: search by SAP column, readable column, or table name
def lookup_column(query: str) -> pd.DataFrame:
    q = query.strip().upper()
    mask = (
        crosswalk["sap_column"].str.upper().str.contains(q, regex=False)
        | crosswalk["readable_column"].str.upper().str.contains(q, regex=False)
        | crosswalk["table_name"].str.upper().str.contains(q, regex=False)
    )
    return crosswalk.loc[mask, ["table_name", "sap_column", "readable_column", "dtype", "ecc_equivalent", "description"]]

lookup_column("KSL")

,table_name,sap_column,readable_column,dtype,ecc_equivalent,description
108,acdoca_actuals,KSL,AMOUNT_USD_REPORTED_UNSIGNED,float64,,Unsigned Reported Group USD amount translated at ACTUAL_AVG FX (S/4HANA carry-over).
109,acdoca_actuals,KSL_BUDGET_FX,AMOUNT_USD_CONSTANT_FX_UNSIGNED,float64,,Unsigned Constant-Currency USD amount translated at annual BUDGET_PLAN FX.
145,acdoca_universal_journal,KSL,AMOUNT_USD_REPORTED_SIGNED,float64,Signed KSL,Signed Reported Group USD amount translated at ACTUAL_AVG FX.
146,acdoca_universal_journal,KSL_BUDGET_FX,AMOUNT_USD_CONSTANT_FX_SIGNED,float64,Signed KSL_BUDGET_FX,Signed Operational Constant-Currency USD translated at BUDGET_PLAN FX (primary ML targ...
166,plan_versions,KSL_BUDGET,PLAN_AMOUNT_USD,float64,,Planned/forecasted Constant-Currency USD spend (= DMBTR_PLAN * PLAN_FX_RATE).


## 2. Side-by-Side Raw SAP vs. Readable Previews

Notice signed vs. unsigned USD clarity (`AMOUNT_USD_REPORTED_UNSIGNED` in `acdoca_actuals` vs. `AMOUNT_USD_REPORTED_SIGNED` in `acdoca_universal_journal`) and collision-free `dim_gl_account` (`GL_ACCOUNT_ID` vs. `GL_ACCOUNT_ID_S4`).

In [4]:
for tbl in ["acdoca_universal_journal", "acdoca_actuals", "dim_cost_center", "dim_gl_account"]:
    df_sap = load_pharma_table(tbl, readable=False)
    df_read = load_pharma_table(tbl, readable=True)
    comparison = pd.DataFrame({
        "sap_column": df_sap.columns,
        "readable_column": df_read.columns,
        "sample_value": [df_read[c].iloc[0] for c in df_read.columns],
    })
    print(f"\n=== {tbl} ({len(df_sap):,} rows x {df_sap.shape[1]} cols) ===")
    print(comparison.to_string(index=False))


=== acdoca_universal_journal (75,268 rows x 32 cols) ===
        sap_column               readable_column    sample_value
            RBUKRS                  COMPANY_CODE            AP30
             WERKS                    PLANT_CODE            IN01
             GJAHR                   FISCAL_YEAR            2022
             RYEAR            LEDGER_FISCAL_YEAR            2022
             POPER                POSTING_PERIOD               1
             BELNR               DOCUMENT_NUMBER      0100000001
             DOCLN              LINE_ITEM_NUMBER          000001
             BLDAT                 DOCUMENT_DATE      2022-01-01
             BUDAT                  POSTING_DATE      2022-01-01
             CPUDT                    ENTRY_DATE      2022-01-01
             BLART                 DOCUMENT_TYPE              SA
             USNAM                  POSTING_USER   BATCH_PREPAID
             RACCT                 GL_ACCOUNT_ID          620070
             RCNTR              


=== acdoca_actuals (75,268 rows x 30 cols) ===
        sap_column                 readable_column    sample_value
             BUKRS                    COMPANY_CODE            AP30
             WERKS                      PLANT_CODE            IN01
             GJAHR                     FISCAL_YEAR            2022
             MONAT                  POSTING_PERIOD               1
             BELNR                 DOCUMENT_NUMBER      0100000001
             BLDAT                   DOCUMENT_DATE      2022-01-01
             BUDAT                    POSTING_DATE      2022-01-01
             CPUDT                      ENTRY_DATE      2022-01-01
             BLART                   DOCUMENT_TYPE              SA
             USNAM                    POSTING_USER   BATCH_PREPAID
             PRCTR                PROFIT_CENTER_ID       PC_ONC_AP
       WBS_ELEMENT                     WBS_ELEMENT  WBS-ONC-P3-009
             LIFNR                       VENDOR_ID VND_SCI_AGILENT
             E

## 3. Critical Join & ML Anti-Leakage Guardrails

### Guardrail A — `DOCLN` vs. `BUZEI` Dtype Mismatch (`LINE_ITEM_NUMBER`)
Both `acdoca_universal_journal.DOCLN` and `acdoca_actuals.BUZEI` / `eval_anomaly_labels.BUZEI` map to `LINE_ITEM_NUMBER`, **but their dtypes differ**:
- `acdoca_universal_journal.DOCLN` is a zero-padded 6-char `string` (`"000001"`, `"000002"`).
- `acdoca_actuals.BUZEI` and `eval_anomaly_labels.BUZEI` are `int64` (`1`, `2`).

Always cast `LINE_ITEM_NUMBER` explicitly (`.astype(int)` or `.astype(str).str.zfill(6)`) before joining `acdoca_universal_journal` with `eval_anomaly_labels` or `acdoca_actuals`.

In [5]:
uj_read = load_pharma_table("acdoca_universal_journal", readable=True)
eval_read = load_pharma_table("eval_anomaly_labels", readable=True)

print("UJ LINE_ITEM_NUMBER dtype & sample:  ", uj_read["LINE_ITEM_NUMBER"].dtype, uj_read["LINE_ITEM_NUMBER"].unique()[:2].tolist())
print("Eval LINE_ITEM_NUMBER dtype & sample:", eval_read["LINE_ITEM_NUMBER"].dtype, eval_read["LINE_ITEM_NUMBER"].unique().tolist())

# Safe join casting LINE_ITEM_NUMBER to int on UJ
uj_for_eval = uj_read.assign(LINE_ITEM_INT=uj_read["LINE_ITEM_NUMBER"].astype(int))
matched = eval_read.merge(
    uj_for_eval,
    left_on=["COMPANY_CODE", "FISCAL_YEAR", "DOCUMENT_NUMBER", "LINE_ITEM_NUMBER"],
    right_on=["COMPANY_CODE", "FISCAL_YEAR", "DOCUMENT_NUMBER", "LINE_ITEM_INT"],
    how="inner",
)
print(f"Matched anomaly labels against UJ: {len(matched)} / {len(eval_read)}")

UJ LINE_ITEM_NUMBER dtype & sample:   str ['000001', '000002']
Eval LINE_ITEM_NUMBER dtype & sample: int64 [1]
Matched anomaly labels against UJ: 104 / 104


### Guardrail B — ML Anti-Leakage Cheat Sheet (`ml_role`)
- **Never use in ML features:** `dim_cost_center.COST_SCALE` (`DO_NOT_USE_IN_MODEL` synthetic sizing knob).
- **Strictly evaluation only (never join into training/inference):** `eval_anomaly_labels` (`EVAL_TARGET_ONLY`, `EVAL_KEY`, `EVAL_METADATA`).
- **Benchmark-only planning rows:** `plan_versions` rows with `VERSION == 2` (`Q1_RF`, `JU`, `Q3_RF`, `NU`) are evaluation benchmarks only (`VERSION == 1` `BP` is safe).
- **Dec 2025 open accruals boundary:** Exclude `GJAHR == 2025 & MONAT == 12` (`FISCAL_YEAR == 2025 & POSTING_PERIOD == 12`) from unreversed-accrual rules (`14` legitimate open `ZA` accruals whose `AB` reversals fall in Jan 2026).

In [6]:
restricted_roles = {"DO_NOT_USE_IN_MODEL", "EVAL_TARGET_ONLY", "EVAL_KEY", "EVAL_METADATA", "BENCHMARK_KEY"}
crosswalk[crosswalk["ml_role"].isin(restricted_roles)][[
    "table_name", "sap_column", "readable_column", "ml_role", "description"
]]

,table_name,sap_column,readable_column,ml_role,description
22,dim_cost_center,COST_SCALE,COST_SCALE,DO_NOT_USE_IN_MODEL,GENERATOR METADATA ONLY - NEVER USE AS AN ML FEATURE. Synthetic relative sizing knob.
161,plan_versions,CYCLE_NAME,CYCLE_NAME,BENCHMARK_KEY,"J&J planning cycle (BP=0+12, Q1_RF=3+9, JU=5+7, Q3_RF=9+3, NU=10+2)."
195,eval_anomaly_labels,BUKRS,COMPANY_CODE,EVAL_KEY,Company code of anomalous document.
196,eval_anomaly_labels,GJAHR,FISCAL_YEAR,EVAL_KEY,Fiscal year of anomalous document.
197,eval_anomaly_labels,MONAT,POSTING_PERIOD,EVAL_METADATA,Posting period (1-12).
198,eval_anomaly_labels,BELNR,DOCUMENT_NUMBER,EVAL_KEY,Chronologically remapped BELNR matching ledger tables.
199,eval_anomaly_labels,BUZEI,LINE_ITEM_NUMBER,EVAL_KEY,Line item number (always 1 = OPEX line).
200,eval_anomaly_labels,KOSTL,COST_CENTER_ID,EVAL_METADATA,Impacted cost center.
201,eval_anomaly_labels,HKONT,GL_ACCOUNT_ID,EVAL_METADATA,Impacted OPEX GL account.
202,eval_anomaly_labels,ANOMALY_TAG,ANOMALY_TAG,EVAL_TARGET_ONLY,Ground-truth finance anomaly class.


## 4. Lossless Round-Trip Verification (All 14 Tables) & Export `docs/sap_column_crosswalk.csv`

In [7]:
for tbl in crosswalk["table_name"].unique():
    df_raw = load_pharma_table(tbl, readable=False)
    df_readable = load_pharma_table(tbl, readable=True)
    assert not df_readable.columns.has_duplicates, f"Duplicate columns in {tbl}"
    df_back = rename_to_sap(df_readable, table_name=tbl, strict=True)
    assert list(df_back.columns) == list(df_raw.columns), f"Round-trip mismatch in {tbl}"

# Explicit table-scoped WBS_ELEMENT reverse-mapping check
drv_back = rename_to_sap(load_pharma_table("fact_opex_drivers", readable=True), "fact_opex_drivers", strict=True)
uj_back = rename_to_sap(load_pharma_table("acdoca_universal_journal", readable=True), "acdoca_universal_journal", strict=True)
assert "WBS_ELEMENT" in drv_back.columns and "PS_PSP_PNR" not in drv_back.columns
assert "PS_PSP_PNR" in uj_back.columns and "WBS_ELEMENT" not in uj_back.columns

crosswalk_path = DOCS_DIR / "sap_column_crosswalk.csv"
crosswalk.to_csv(crosswalk_path, index=False)
print(f"Verified lossless round-trip on all {crosswalk['table_name'].nunique()} tables.")
print(f"Wrote {crosswalk_path} ({len(crosswalk)} rows).")

Verified lossless round-trip on all 14 tables.
Wrote D:\Code Projects\opex_forecast\docs\sap_column_crosswalk.csv (206 rows).
